# Part 8 · Notebook 12 — Risk parity, HRP, Black–Litterman and CVaR

**Sessions:** S23 (Risk parity, HRP, Black–Litterman & CVaR) · S24 (Integration & M5a release) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Compute risk contributions, and build a portfolio where they are equal.
2. Write the recursive bisection step of Hierarchical Risk Parity.
3. Blend equilibrium returns with a view using Black–Litterman.
4. Compare seven allocators walk-forward, through three crises.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. Risk contributions

Capital weights hide where the risk is. The share of portfolio variance coming from asset `i` is `w_i·(Σw)_i / (w'Σw)`; the shares sum to 1. Six assets from notebook 09.

In [ ]:
A = p.asset_returns()
cov = A.cov().to_numpy() * 252

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def risk_contributions(w, cov):
    w = np.asarray(w, dtype=float)
    return ...                                    # ✍️

w6040 = np.array([0.35, 0.25, 0.25, 0.05, 0.05, 0.05])
w_rp = p.risk_parity(cov)
mine = [p.attempt(risk_contributions, w6040, cov), p.attempt(risk_contributions, w_rp, cov)]
mine = p.check("risk_contributions", mine, [p.risk_contributions(w6040, cov), p.risk_contributions(w_rp, cov)])
pd.DataFrame({"60/40 weights": w6040, "60/40 risk": mine[0], "risk-parity weights": w_rp, "risk-parity risk": mine[1]}, index=A.columns).round(3)

Risk parity equalizes the contributions by putting most of the capital into the least volatile asset; to reach a useful return it is usually levered (or volatility-targeted, notebook 10).

## 2. Hierarchical Risk Parity

HRP (López de Prado, 2016) avoids inverting the covariance matrix. Cluster the assets by correlation and order them so similar assets sit together (`p.hrp_order`); then **bisect** that order recursively: at each split, the variance of each half's inverse-variance portfolio (`p.cluster_variance`) decides how to share the weight, `α = 1 − v_a / (v_a + v_b)` to the left half.

In [ ]:
print("quasi-diagonal order:", p.hrp_order(A))

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def hrp(returns):
    cov_ = returns.cov()
    order = p.hrp_order(returns)
    w = pd.Series(1.0, index=order)
    clusters = [order]
    while clusters:
        clusters = [c[i:j] for c in clusters for i, j in ((0, len(c) // 2), (len(c) // 2, len(c))) if len(c) > 1]
        for a, b in zip(clusters[::2], clusters[1::2]):
            va, vb = p.cluster_variance(cov_, a), p.cluster_variance(cov_, b)
            alpha = ...                           # ✍️ the left half's share
            w[a] *= alpha
            w[b] *= 1 - alpha
    return w.reindex(returns.columns)

mine = p.attempt(hrp, A)
mine = p.check("hrp", mine, p.hrp(A))
pd.DataFrame({"HRP": mine, "risk parity": w_rp, "inverse vol": p.inverse_vol_weights(A)}, index=A.columns).round(3)

## 3. Black–Litterman

Instead of noisy sample means, start from the returns the market portfolio **implies**, `π = δ·Σ·w_mkt`, and tilt them by your views. A view is a row of `P` (which assets) with an expected value in `Q`, and uncertainty `Ω` (default `diag(P·τΣ·P')`). The posterior is `μ = [(τΣ)⁻¹ + P'Ω⁻¹P]⁻¹ · [(τΣ)⁻¹π + P'Ω⁻¹Q]`. Use `np.linalg.inv` and `np.linalg.solve`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def black_litterman(cov, w_mkt, P, Q, delta=2.5, tau=0.05):
    S_ = np.asarray(cov, dtype=float)
    pi = delta * S_ @ np.asarray(w_mkt, dtype=float)
    P, Q = np.atleast_2d(np.asarray(P, dtype=float)), np.asarray(Q, dtype=float)
    tS = tau * S_
    om = np.diag(np.diag(P @ tS @ P.T))
    A_ = ...                                      # ✍️ (τΣ)⁻¹ + P'Ω⁻¹P
    b = ...                                       # ✍️ (τΣ)⁻¹π + P'Ω⁻¹Q
    return np.linalg.solve(A_, b)

P = [[-1, 1, 0, 0, 0, 0]]                         # view: international equities beat US equities…
Q = [0.02]                                        # …by 2% a year
mine = p.attempt(black_litterman, cov, w6040, P, Q)
mine = p.check("black_litterman", mine, p.black_litterman(cov, w6040, P=P, Q=Q))
pd.DataFrame({"equilibrium π": p.black_litterman(cov, w6040), "with the view": mine}, index=A.columns).mul(100).round(2)

The view moves the two equity markets apart, and moves the correlated assets (commodities) along with them, by an amount set by the view's confidence. The result is a sane input for an optimizer, unlike sample means.

## 4. Minimum CVaR, and the comparison

`p.min_cvar` solves the Rockafellar–Uryasev linear program: the long-only weights that minimize the historical 95% CVaR. Now compare seven allocators **walk-forward**: refit monthly on the previous year, hold the next month, through all three crises.

In [ ]:
alloc = {"1/N": lambda X: np.full(X.shape[1], 1 / X.shape[1]),
         "inverse vol": lambda X: p.inverse_vol_weights(X).to_numpy(),
         "min variance (sample)": lambda X: p.min_variance(X.cov().to_numpy()),
         "min variance (shrunk)": lambda X: p.min_variance(p.ledoit_wolf(X)),
         "risk parity": lambda X: p.risk_parity(X.cov().to_numpy()),
         "HRP": lambda X: p.hrp(X).to_numpy(),
         "min CVaR 95%": lambda X: p.min_cvar(X, 0.95)}
rows, curves = {}, {}
for name, f in alloc.items():
    res = p.walk_forward_allocation(A, f)
    r = res["returns"]
    curves[name] = (1 + r).cumprod()
    rows[name] = {"OOS return": r.mean() * 252, "OOS vol": r.std() * np.sqrt(252), "Sharpe": p.sharpe(r),
                  "max drawdown": p.max_drawdown(r)[0], "turnover": res["turnover"]}
table = pd.DataFrame(rows).T
display(table.round(3))
fig, ax = plt.subplots(figsize=(11, 4))
for name, eq in curves.items():
    ax.plot(eq.index, eq, lw=1.2, label=name)
for s in (600, 1700, 2300):
    ax.axvspan(A.index[s], A.index[s + 59], color=p.PALETTE[7], alpha=0.1)
ax.set_title("Walk-forward allocators (shaded: crises)"); ax.legend(ncol=2, fontsize=8); plt.show()

Read the table in two columns. By Sharpe and drawdown the risk-based allocators win, because they load up on the steadiest assets; by return 1/N wins. A fair comparison scales every portfolio to the **same volatility** first (notebook 10), and counts turnover as cost. Whichever wins must win walk-forward, not in-sample (common mistake #12).

## Wrap-up

* Look at risk contributions, not capital weights.
* HRP and risk parity avoid inverting noisy matrices; Black–Litterman gives sane expected returns; CVaR optimizes the tail.
* The allocator itself is a parameter: choose it walk-forward.
* Graded versions: `labs/part08/week30_portfolio` and Clinic W6 (allocator comparison and recommendation).